In [ ]:
import random

class ExamSeatingGA:
    def __init__(self, students, grid_size, pop_size=100, generations=200, crossover_rate=0.8, mutation_rate=0.1):
        """
        students: List of tuples (student_id, subject_code)
        grid_size: Tuple (rows, cols)
        """
        self.rows, self.cols = grid_size
        self.total_seats = self.rows * self.cols
        self.pop_size = pop_size
        self.generations = generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate

        if len(students) > self.total_seats:
            raise ValueError("Number of students exceeds the total number of available seats.")

        # Pad the student list with None for empty seats to fill the grid
        self.seating_pool = students + [None] * (self.total_seats - len(students))

    def calculate_fitness(self, chromosome):
        """
        Fitness = 1 / (1 + adjacent same-subject pairs)
        Checks horizontal and vertical adjacencies in the 2D grid.
        """
        adjacent_conflicts = 0

        # Reshape linear chromosome into 2D grid representation for adjacency checking
        grid = [chromosome[i * self.cols:(i + 1) * self.cols] for i in range(self.rows)]

        for r in range(self.rows):
            for c in range(self.cols):
                current_idx = grid[r][c]
                current_student = self.seating_pool[current_idx]

                # Skip empty seats
                if current_student is None:
                    continue

                current_subject = current_student[1]

                # Check right neighbor
                if c + 1 < self.cols:
                    right_idx = grid[r][c + 1]
                    right_student = self.seating_pool[right_idx]
                    if right_student and right_student[1] == current_subject:
                        adjacent_conflicts += 1

                # Check down neighbor
                if r + 1 < self.rows:
                    down_idx = grid[r + 1][c]
                    down_student = self.seating_pool[down_idx]
                    if down_student and down_student[1] == current_subject:
                        adjacent_conflicts += 1

        return 1.0 / (1.0 + adjacent_conflicts)

    def order_crossover(self, parent1, parent2):
        """
        Implements Order Crossover (OX1) for permutation chromosomes.
        """
        if random.random() > self.crossover_rate:
            return parent1.copy(), parent2.copy()

        size = len(parent1)
        start, end = sorted(random.sample(range(size), 2))

        def fill_child(p1, p2):
            child = [None] * size
            # Copy matching segment from first parent
            child[start:end+1] = p1[start:end+1]

            # Fill the remaining positions using the order from the second parent
            p2_idx = 0
            for i in range(size):
                if child[i] is None:
                    while p2[p2_idx] in child:
                        p2_idx += 1
                    child[i] = p2[p2_idx]
            return child

        return fill_child(parent1, parent2), fill_child(parent2, parent1)

    def swap_mutation(self, chromosome):
        """
        Implements Swap Mutation by swapping two random positions.
        """
        if random.random() < self.mutation_rate:
            idx1, idx2 = random.sample(range(len(chromosome)), 2)
            chromosome[idx1], chromosome[idx2] = chromosome[idx2], chromosome[idx1]
        return chromosome

    def selection(self, population, fitnesses):
        """
        Selects a parent using Roulette Wheel Selection.
        """
        total_fitness = sum(fitnesses)
        pick = random.uniform(0, total_fitness)
        current = 0
        for ind, fit in zip(population, fitnesses):
            current += fit
            if current > pick:
                return ind
        return population[-1]

    def solve(self):
        # Initialize population as random permutations of indices
        base_chromosome = list(range(self.total_seats))
        population = [random.sample(base_chromosome, self.total_seats) for _ in range(self.pop_size)]

        best_chromosome = None
        best_fitness = -1

        for gen in range(self.generations):
            fitnesses = [self.calculate_fitness(ind) for ind in population]

            # Track the absolute best solution found
            for ind, fit in zip(population, fitnesses):
                if fit > best_fitness:
                    best_fitness = fit
                    best_chromosome = ind.copy()

            # If an ideal arrangement (no conflicts) is found, stop early
            if best_fitness == 1.0:
                print(f"Optimal solution found at generation {gen}!")
                break

            new_population = []
            while len(new_population) < self.pop_size:
                p1 = self.selection(population, fitnesses)
                p2 = self.selection(population, fitnesses)

                c1, c2 = self.order_crossover(p1, p2)

                new_population.append(self.swap_mutation(c1))
                if len(new_population) < self.pop_size:
                    new_population.append(self.swap_mutation(c2))

            population = new_population

        self.print_seating_chart(best_chromosome, best_fitness)

    def print_seating_chart(self, chromosome, fitness):
        """
        Outputs the formatted seating grid.
        """
        print(f"\nFinal Seating Chart (Fitness: {fitness:.4f}):")
        print("-" * (self.cols * 15 + 1))
        for r in range(self.rows):
            row_str = "| "
            for c in range(self.cols):
                idx = chromosome[r * self.cols + c]
                student = self.seating_pool[idx]
                if student:
                    row_str += f"{student[0]}({student[1]}):".ljust(13) + " | "
                else:
                    row_str += "Empty".ljust(13) + " | "
            print(row_str)
            print("-" * (self.cols * 15 + 1))

# --- Example Usage ---
if __name__ == "__main__":
    # Sample input: (Student_ID, Subject_Code)
    student_list = [
        ('S01', 'MATH'), ('S02', 'MATH'), ('S03', 'MATH'),
        ('S04', 'PHYS'), ('S05', 'PHYS'), ('S06', 'PHYS'),
        ('S07', 'CHEM'), ('S08', 'CHEM'), ('S09', 'CHEM'),
        ('S10', 'BIO'), ('S11', 'BIO'), ('S12', 'BIO')
    ]

    # 4 rows by 4 columns grid (16 seats total, leaving 4 empty spots)
    grid = (4, 4)

    ga = ExamSeatingGA(student_list, grid, pop_size=50, generations=300)
    ga.solve()


Optimal solution found at generation 1!

Final Seating Chart (Fitness: 1.0000):
-------------------------------------------------------------
| S08(CHEM):    | S11(BIO):     | Empty         | S01(MATH):    | 
-------------------------------------------------------------
| Empty         | S02(MATH):    | S06(PHYS):    | S09(CHEM):    | 
-------------------------------------------------------------
| Empty         | Empty         | S07(CHEM):    | S12(BIO):     | 
-------------------------------------------------------------
| S05(PHYS):    | S10(BIO):     | S04(PHYS):    | S03(MATH):    | 
-------------------------------------------------------------
